In [ ]:
import os
import asyncio
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm_asyncio
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

from datasets import load_dataset

ds = load_dataset("jrosseruk/cocktails-with-instructions")

# Convert the 'train' split (or another available split) into a pandas DataFrame
df = ds["train"].to_pandas()

In [ ]:
df.head()

In [ ]:
[ing[1] for ing in eval(df.iloc[0]["ingredients"])]

In [ ]:
import ast
from collections import Counter

N = 50

# Parse the ingredients column from string to actual lists
df['ingredients_list'] = df['ingredients'].apply(lambda x: [ing[1] for ing in eval(x)])

# Flatten all ingredients and count them
all_ingredients = []
for ingredients in df['ingredients_list']:
    all_ingredients.extend(ingredients)

ingredient_counts = Counter(all_ingredients)

# Get the N most common ingredients
top_N_ingredients = set([ing for ing, count in ingredient_counts.most_common(N)])
print(f"Top N ingredients: {len(top_N_ingredients)}")
print(ingredient_counts.most_common(N))  # Show top 20 for preview

In [ ]:
# Filter recipes that only contain ingredients from the top N
def all_ingredients_in_top_N(ingredients_list):
    return all(ing in top_N_ingredients for ing in ingredients_list)

df_filtered = df[df['ingredients_list'].apply(all_ingredients_in_top_N)].copy()

print(f"Original recipes: {len(df)}")
print(f"Filtered recipes (only top N ingredients): {len(df_filtered)}")
print(f"\nFiltered dataset preview:")
df_filtered.head()